# Type 2 Diabetes Cohort Definition with ATHENA Concepts

This notebook demonstrates how to:
1. Use the `athena-client` package to search for Type 2 Diabetes concepts
2. Efficiently convert ATHENA concepts to CIRCE Concept objects
3. Build concept sets from ATHENA search results
4. Create a cohort definition using CIRCE Python
5. Generate SQL for OMOP CDM execution
6. Validate the cohort definition

## Requirements

```bash
pip install ohdsi-circe athena-client pandas
```


## Step 1: Import Required Libraries


In [1]:
# Core libraries
import pandas as pd
from IPython.display import display, Markdown

# ATHENA client for concept lookup
try:
    from athena_client import AthenaClient
    ATHENA_AVAILABLE = True
except ImportError:
    print("⚠️ athena-client not installed. Install with: pip install athena-client")
    ATHENA_AVAILABLE = False

# CIRCE Python for cohort definitions
from circe.cohortdefinition import (
    CohortExpression, PrimaryCriteria, ConditionOccurrence,
    ObservationFilter, ResultLimit
)
from circe.vocabulary import ConceptSet, ConceptSetExpression, ConceptSetItem, Concept
from circe.api import build_cohort_query
from circe.check import Checker

print("✓ All libraries imported successfully")


✓ All libraries imported successfully


## Step 2: Helper Functions for ATHENA → CIRCE Conversion

Create efficient conversion functions to transform athena-client objects into CIRCE Concept and ConceptSet objects.


In [2]:
def athena_to_circe_concept(athena_concept: dict) -> Concept:
    """
    Convert an athena-client concept dictionary to a CIRCE Concept object.
    
    Args:
        athena_concept: Dictionary from athena-client search results
        
    Returns:
        CIRCE Concept object
    """
    # Handle standardConcept enum or string
    standard_concept = athena_concept.get('standardConcept')
    if hasattr(standard_concept, 'value'):
        standard_concept = standard_concept.value
    
    # Convert "Standard" to "S", "Classification" to "C"
    if standard_concept == "Standard":
        standard_concept = "S"
    elif standard_concept == "Classification":
        standard_concept = "C"
    else:
        standard_concept = None
    
    return Concept(
        concept_id=athena_concept['id'],
        concept_name=athena_concept['name'],
        domain_id=athena_concept['domain'],
        vocabulary_id=athena_concept['vocabulary'],
        concept_class_id=athena_concept['className'],
        standard_concept=standard_concept,
        concept_code=athena_concept['code']
    )


def athena_concepts_to_circe_concept_set(
    athena_concepts: list,
    concept_set_id: int,
    concept_set_name: str,
    include_descendants: bool = True,
    include_mapped: bool = False
) -> ConceptSet:
    """
    Convert a list of athena-client concepts to a CIRCE ConceptSet.
    
    Args:
        athena_concepts: List of concept dicts from athena-client
        concept_set_id: ID for the concept set
        concept_set_name: Name for the concept set
        include_descendants: Whether to include descendant concepts
        include_mapped: Whether to include mapped concepts
        
    Returns:
        CIRCE ConceptSet object
    """
    # Convert each athena concept to CIRCE concept and create concept set items
    items = []
    for athena_concept in athena_concepts:
        circe_concept = athena_to_circe_concept(athena_concept)
        item = ConceptSetItem(
            concept=circe_concept,
            include_descendants=include_descendants,
            is_excluded=False
        )
        items.append(item)
    
    # Create concept set expression (requires top-level include flags)
    expression = ConceptSetExpression(
        items=items,
        is_excluded=False,                       # Expression is not excluded
        include_mapped=include_mapped,           # Include mapped concepts
        include_descendants=include_descendants  # Include descendants
    )
    
    # Create and return concept set
    return ConceptSet(
        id=concept_set_id,
        name=concept_set_name,
        expression=expression
    )


print("✓ Helper functions defined")


✓ Helper functions defined


## Step 3: Search ATHENA for Type 2 Diabetes Concepts

Use the athena-client to search for Type 2 Diabetes concepts in SNOMED vocabulary.


In [3]:
if ATHENA_AVAILABLE:
    # Initialize ATHENA client
    client = AthenaClient()
    
    # Search for Type 2 Diabetes concepts
    print("Searching ATHENA for Type 2 Diabetes concepts...")
    result = client.search(
        "Type 2 diabetes mellitus",
        vocabulary="SNOMED",
        domain="Condition",
        standard_concept="Standard",
        size=10
    )
    
    # Convert to list of dicts
    athena_concepts = result.to_list()
    
    # Display search results
    print(f"\n✓ Found {len(athena_concepts)} concepts:")
    df = pd.DataFrame(athena_concepts)
    display(df[['id', 'name', 'vocabulary', 'className', 'code']].head(10))
    
    # Select the primary concept (first result - exact match)
    primary_athena_concept = athena_concepts[0]
    
    print(f"\n✓ Selected primary concept:")
    print(f"  ID: {primary_athena_concept['id']}")
    print(f"  Name: {primary_athena_concept['name']}")
    print(f"  Vocabulary: {primary_athena_concept['vocabulary']}")
else:
    # Fallback: Use known Type 2 Diabetes concept
    print("⚠️ ATHENA client not available. Using known concept.")
    athena_concepts = [{
        'id': 201826,
        'name': 'Type 2 diabetes mellitus',
        'domain': 'Condition',
        'vocabulary': 'SNOMED',
        'className': 'Disorder',
        'standardConcept': 'Standard',
        'code': '44054006',
        'invalidReason': 'Valid',
        'score': None
    }]
    primary_athena_concept = athena_concepts[0]
    print(f"Using concept ID: {primary_athena_concept['id']} - {primary_athena_concept['name']}")


Searching ATHENA for Type 2 Diabetes concepts...
ℹ️  Note: Query 'Type 2 diabetes mellitus' matches many concepts (estimated 5,000+), but only 10 will be downloaded.

✓ Found 10 concepts:


,id,name,vocabulary,className,code
0,201826,Type 2 diabetes mellitus,SNOMED,Disorder,44054006
1,4063043,Pre-existing type 2 diabetes mellitus,SNOMED,Disorder,199230006
2,40482801,Uncontrolled type 2 diabetes mellitus,SNOMED,Disorder,443694000
3,40483315,Brittle type 2 diabetes mellitus,SNOMED,Disorder,445353002
4,40545711,Insulin treated Type 2 diabetes mellitus,SNOMED,Disorder,366909003
5,40485020,Well controlled type 2 diabetes mellitus,SNOMED,Disorder,444110003
6,4129519,Pregnancy and type 2 diabetes mellitus,SNOMED,Disorder,237627000
7,3569411,Type 2 diabetes mellitus with gastroparesis,SNOMED,Disorder,198671000000109
8,4130162,Insulin treated type 2 diabetes mellitus,SNOMED,Disorder,237599002
9,37016349,Hyperglycemia due to type 2 diabetes mellitus,SNOMED,Disorder,368051000119109



✓ Selected primary concept:
  ID: 201826
  Name: Type 2 diabetes mellitus
  Vocabulary: SNOMED


## Step 4: Convert ATHENA Concepts to CIRCE ConceptSet

Use our helper function to efficiently convert the ATHENA search results to a CIRCE ConceptSet.


In [4]:
# Convert just the primary concept to a CIRCE concept set
# Include descendants to capture all Type 2 Diabetes subtypes
t2dm_concept_set = athena_concepts_to_circe_concept_set(
    athena_concepts=[primary_athena_concept],  # Just the main concept
    concept_set_id=1,
    concept_set_name="Type 2 Diabetes Mellitus",
    include_descendants=True  # Include all subtypes
)

print("✓ Concept set created:")
print(f"  ID: {t2dm_concept_set.id}")
print(f"  Name: {t2dm_concept_set.name}")
print(f"  Number of concepts: {len(t2dm_concept_set.expression.items)}")
print(f"  Primary concept: {t2dm_concept_set.expression.items[0].concept.concept_id} - {t2dm_concept_set.expression.items[0].concept.concept_name}")
print(f"  Include descendants: {t2dm_concept_set.expression.items[0].include_descendants}")


✓ Concept set created:
  ID: 1
  Name: Type 2 Diabetes Mellitus
  Number of concepts: 1
  Primary concept: 201826 - Type 2 diabetes mellitus
  Include descendants: True


## Step 5: Define Primary Criteria

Define the cohort entry criteria: first occurrence of Type 2 Diabetes condition.


In [5]:
# Create primary criteria: First Type 2 Diabetes diagnosis
primary_criteria = PrimaryCriteria(
    criteria_list=[
        ConditionOccurrence(
            codeset_id=1,                    # References concept set ID 1
            first=True,                      # Only the first occurrence
            condition_type_exclude=False     # Include all condition types
        )
    ],
    observation_window=ObservationFilter(
        prior_days=365,   # Patient must have 1 year observation starting on or before diagnosis
        post_days=1     # Patient must have observation on or after diagnosis
    ),
    primary_limit=ResultLimit(
        type="All"      # Include all qualifying events
    )
)

print("✓ Primary criteria defined:")
print(f"  Criteria Type: Condition Occurrence")
print(f"  Codeset ID: {primary_criteria.criteria_list[0].codeset_id}")
print(f"  First Occurrence Only: {primary_criteria.criteria_list[0].first}")


✓ Primary criteria defined:
  Criteria Type: Condition Occurrence
  Codeset ID: 1
  First Occurrence Only: True


## Step 6: Create Complete Cohort Expression

Combine concept sets and primary criteria into a complete cohort definition.


In [8]:
# Create the cohort expression
cohort = CohortExpression(
    title="Type 2 Diabetes Mellitus Patients",
    concept_sets=[t2dm_concept_set],
    primary_criteria=primary_criteria
)

print("✓ Cohort expression created:")
print(f"  Title: {cohort.title}")
print(f"  Number of Concept Sets: {len(cohort.concept_sets)}")
print(f"  Primary Criteria Type: {type(primary_criteria.criteria_list[0]).__name__}")


✓ Cohort expression created:
  Title: Type 2 Diabetes Mellitus Patients
  Number of Concept Sets: 1
  Primary Criteria Type: ConditionOccurrence


## Step 7: Validate Cohort Definition

Check for potential issues before generating SQL.


In [9]:
# Validate the cohort
checker = Checker()
warnings = checker.check(cohort)

if not warnings:
    print("✓ Cohort definition is valid with no warnings!")
else:
    print(f"⚠️ Validation found {len(warnings)} issues:")
    for warning in warnings:
        print(f"  {warning.to_message()}")


⚠️ Validation found 1 issues:
  It's not specified what type of records to look for in condition occurrence at initial event


## Step 8: Generate SQL Query

Generate the SQL query that can be executed against an OMOP CDM database.


In [10]:
# Generate SQL with your database schema names
from circe.cohortdefinition import BuildExpressionQueryOptions

options = BuildExpressionQueryOptions()
options.cdm_schema = "my_cdm_schema"        # Replace with your CDM schema name
options.vocabulary_schema = "my_vocab_schema"    # Replace with your vocabulary schema name
options.target_table = "cohort"
options.cohort_id = 1                        # Cohort ID for the results table

sql = build_cohort_query(cohort, options)

print(f"✓ SQL generated ({len(sql)} characters)")
print("\nFirst 1000 characters of SQL:")
print("=" * 80)
print(sql[:1000])
print("...")
print("=" * 80)


✓ SQL generated (3901 characters)

First 1000 characters of SQL:

    
    IF OBJECT_ID('tempdb..#Codesets', 'U') IS NOT NULL
    DROP TABLE #Codesets;

    CREATE TABLE #Codesets (
      codeset_id int NOT NULL,
      concept_id bigint NOT NULL
    );

    INSERT INTO #Codesets (codeset_id, concept_id)
SELECT 1 as codeset_id, c.concept_id FROM (
    SELECT DISTINCT concept_id
    FROM (
    SELECT concept_id 
    FROM my_vocab_schema.CONCEPT 
    WHERE concept_id IN (201826)
     UNION 
    SELECT ca.descendant_concept_id as concept_id
    FROM my_vocab_schema.CONCEPT_ANCESTOR ca
    WHERE ca.ancestor_concept_id IN (201826)
    ) concept_set
    ) C;
    

    
SELECT event_id, person_id, start_date, end_date, op_start_date, op_end_date, visit_occurrence_id
INTO #qualified_events
FROM 
(
  select pe.event_id, pe.person_id, pe.start_date, pe.end_date, pe.op_start_date, pe.op_end_date, row_number() over (partition by pe.person_id order by pe.start_date ASC) as ordinal, cast(pe.visit_occ

## Step 9: Save Outputs

Save the cohort definition (JSON) and SQL query to files.


In [ ]:
# Save cohort definition as JSON
cohort_json = cohort.model_dump_json(indent=2, exclude_none=True, by_alias=False)

with open('type2_diabetes_cohort.json', 'w') as f:
    f.write(cohort_json)
print("✓ Cohort definition saved to: type2_diabetes_cohort.json")

# Save SQL query
with open('type2_diabetes_cohort.sql', 'w') as f:
    f.write(sql)
print("✓ SQL query saved to: type2_diabetes_cohort.sql")

# Display summary
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"Cohort Title: {cohort.title}")
print(f"Concept Sets: {len(cohort.concept_sets)}")
print(f"  - {t2dm_concept_set.name} (ID: {t2dm_concept_set.id})")
print(f"Primary Criteria: First Condition Occurrence")
print(f"SQL Length: {len(sql)} characters")
print(f"Validation: {'✓ PASSED' if not warnings else f'⚠️  {len(warnings)} warnings'}")
print(f"{'='*80}")


✓ Cohort definition saved to: type2_diabetes_cohort.json
✓ SQL query saved to: type2_diabetes_cohort.sql

SUMMARY
Cohort Title: Type 2 Diabetes Mellitus Patients
Concept Sets: 1
  - Type 2 Diabetes Mellitus (ID: 1)
Primary Criteria: First Condition Occurrence
SQL Length: 3901 characters
Validation: ⚠️  1 warnings


## Bonus: Building More Complex Concept Sets

You can also search for multiple related concepts and combine them into a single concept set.


In [10]:
if ATHENA_AVAILABLE:
    # Example: Search for Metformin (common Type 2 Diabetes medication)
    print("Searching for Metformin concepts...")
    metformin_result = client.search(
        "Metformin",
        vocabulary="RxNorm",
        domain="Drug",
        standard_concept="Standard",
        size=5
    )
    
    metformin_concepts = metformin_result.to_list()
    
    # Convert to CIRCE concept set
    metformin_concept_set = athena_concepts_to_circe_concept_set(
        athena_concepts=metformin_concepts[:3],  # Take top 3 results
        concept_set_id=2,
        concept_set_name="Metformin",
        include_descendants=True
    )
    
    print(f"\n✓ Metformin concept set created:")
    print(f"  ID: {metformin_concept_set.id}")
    print(f"  Name: {metformin_concept_set.name}")
    print(f"  Number of concepts: {len(metformin_concept_set.expression.items)}")
    
    # Display the concepts
    for item in metformin_concept_set.expression.items:
        c = item.concept
        print(f"    - {c.concept_id}: {c.concept_name}")
else:
    print("⚠️ ATHENA client not available. Skipping Metformin example.")


Searching for Metformin concepts...
ℹ️  Note: Query 'Metformin' matches many concepts (estimated 1,000+), but only 5 will be downloaded.

✓ Metformin concept set created:
  ID: 2
  Name: Metformin
  Number of concepts: 3
    - 1503297: metformin
    - 36223145: metformin Pill
    - 19055308: metformin hydrochloride


## Optional: Specifying Condition Types (to suppress INFO warning)

If you want to be explicit about which condition record types to include (and suppress the validation INFO message), you can specify `condition_type`. By default, leaving it `None` means "accept all types".


In [14]:
# Example: Create a more specific cohort that only accepts EHR records
# (This will have zero validation warnings)

from circe.cohortdefinition import ConditionOccurrence

# Common condition type concepts (from OMOP CDM concept table)
# These specify where the condition diagnosis came from:
# - 32020: EHR problem list entry
# - 32817: EHR
# - 32809: Case Report Form
# - 32810: Patient self-report

ehr_condition_types = [
    Concept(
        concept_id=32817,
        concept_name="EHR",
        domain_id="Type Concept",
        vocabulary_id="Condition Type",
        concept_class_id="Condition Type",
        standard_concept="S",
        concept_code="OMOP4976890"
    )
]

# Create criteria with explicit condition type
specific_criteria = PrimaryCriteria(
    criteria_list=[
        ConditionOccurrence(
            codeset_id=1,
            first=True,
            condition_type=ehr_condition_types,  # Specify EHR records only
            condition_type_exclude=False
        )
    ],
    observation_window=ObservationFilter(prior_days=0, post_days=0),
    primary_limit=ResultLimit(type="All")
)

# Create cohort with specific criteria
specific_cohort = CohortExpression(
    title="Type 2 Diabetes (EHR Only)",
    concept_sets=[t2dm_concept_set],
    primary_criteria=specific_criteria
)

# Validate - should have zero warnings now
checker2 = Checker()
warnings2 = checker2.check(specific_cohort)

if not warnings2:
    print("✓ Specific cohort has NO warnings!")
else:
    print(f"⚠️ Found {len(warnings2)} warnings:")
    for w in warnings2:
        print(f"  [{w.severity.name}] {w.message}")

print(f"\nℹ️  Note: Using condition_type is optional. The original cohort")
print(f"   (without condition_type) will work perfectly fine - it just accepts")
print(f"   ALL condition types, which is usually what you want.")


✓ Specific cohort has NO warnings!

ℹ️  Note: Using condition_type is optional. The original cohort
   (without condition_type) will work perfectly fine - it just accepts
   ALL condition types, which is usually what you want.


## Summary

This notebook demonstrated:

1. ✓ Using **athena-client** (`AthenaClient`) to search for OMOP concepts
2. ✓ Efficient conversion functions from ATHENA to CIRCE objects
3. ✓ Building concept sets from ATHENA search results
4. ✓ Creating a cohort definition with CIRCE Python
5. ✓ Validating the cohort definition
6. ✓ Generating SQL for OMOP CDM execution
7. ✓ Exporting cohort definition as JSON

### Key Conversion Functions

- `athena_to_circe_concept()` - Convert single ATHENA concept to CIRCE Concept
- `athena_concepts_to_circe_concept_set()` - Convert multiple ATHENA concepts to CIRCE ConceptSet

### Files Created

- `type2_diabetes_cohort.json` - Cohort definition (Atlas-compatible)
- `type2_diabetes_cohort.sql` - SQL query for OMOP CDM

### Additional Resources

- **athena-client docs**: https://github.com/OHDSI/athena-client
- **CIRCE Python docs**: https://ohdsi-circe.readthedocs.io/
- **OHDSI ATHENA**: https://athena.ohdsi.org/
- **OMOP CDM**: https://ohdsi.github.io/CommonDataModel/
